1장. AI와 함께하는 데이터 분석의 시작

LLM 기반 데이터 분석 실무 입문 과정의 실습 노트북입니다.

학습 목표

주요 개념과 분석 흐름을 이해합니다.
제공된 실습 데이터를 불러와 기본 구조를 확인합니다.
LLM을 분석 보조 도구로 활용하는 방법을 익힙니다.

실습 배경

이번 장의 내용을 실습하면서 데이터 로드, 탐색, 전처리, 시각화의 기본 흐름을 확인합니다.

In [5]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data/raw')
sns.set_theme(style='whitegrid')

데이터 불러오기 기본 설정

샘플 데이터를 불러오기 위한 기본 설정입니다. 필요에 따라 아래 코드를 수정해 실행합니다.

In [6]:
# 예시: 필요한 데이터 파일을 읽고 처음 몇 행을 확인합니다.
customers = pd.read_csv(DATA_DIR / 'customers.csv')
customers.head()

,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-08-21
1,2,김정호,F,32,대구,2026-01-04
2,3,이경수,F,61,성남,2024-08-14
3,4,조영호,F,55,울산,2026-06-15
4,5,이예원,F,19,부산,2024-11-15


In [8]:
#분석 코드
import pandas as pd

# 1. 파일 불러오기
customers = pd.read_csv(DATA_DIR / 'customers.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')

# 2. 분석 기간과 완료 주문만 선택
orders["order_date"] = pd.to_datetime(orders["order_date"])

completed_orders = orders[
    (orders["order_date"] >= "2025-09-01") &
    (orders["order_date"] <= "2026-08-31") &
    (orders["order_status"] == "completed")
]

# 3. 주문·주문상세·상품 정보 연결
data = (
    completed_orders[["order_id", "payment_method"]]
    .merge(order_items, on="order_id")
    .merge(products[["product_id", "category"]], on="product_id")
)

# 4. 주문상품별 매출 계산
data["sales"] = data["quantity"] * data["unit_price"]

# 5. 주문별 결제금액과 구매 수량 계산
order_summary = (
    data.groupby(["order_id", "payment_method"])
    .agg(
        order_amount=("sales", "sum"),
        item_quantity=("quantity", "sum")
    )
    .reset_index()
)

# 6. 결제수단별 평균 결제금액과 평균 구매 수량
payment_summary = (
    order_summary.groupby("payment_method")
    .agg(
        completed_orders=("order_id", "count"),
        avg_order_amount=("order_amount", "mean"),
        avg_item_quantity=("item_quantity", "mean")
    )
    .round(1)
)

print("[결제수단별 완료 주문 요약]")
print(payment_summary)

# 7. 결제수단별 카테고리 매출 비중
category_summary = (
    data.groupby(["payment_method", "category"])["sales"]
    .sum()
    .reset_index()
)

category_summary["category_sales_share(%)"] = (
    category_summary["sales"]
    / category_summary.groupby("payment_method")["sales"].transform("sum")
    * 100
).round(1)

print("\n[결제수단별 카테고리 매출 비중]")
print(
    category_summary.sort_values(
        ["payment_method", "category_sales_share(%)"],
        ascending=[True, False]
    )
)

[결제수단별 완료 주문 요약]
                completed_orders  avg_order_amount  avg_item_quantity
payment_method                                                       
bank_transfer                 45          763155.6                7.2
card                          38          803710.5                8.1
kakao_pay                     47          792744.7                7.8
naver_pay                     51          852941.2                8.1

[결제수단별 카테고리 매출 비중]
   payment_method category    sales  category_sales_share(%)
1   bank_transfer       뷰티  7757000                     22.6
3   bank_transfer      스포츠  7481000                     21.8
2   bank_transfer     생활용품  4655000                     13.6
5   bank_transfer     전자기기  4325000                     12.6
0   bank_transfer       도서  3909000                     11.4
4   bank_transfer       식품  3718000                     10.8
6   bank_transfer       패션  2497000                      7.3
12           card     전자기기  5935000                    

실습 진행

이제 데이터를 직접 다루며 간단한 분석 결과를 확인합니다. 필요한 코드를 자유롭게 추가하세요.

LLM 프롬프트 활용 예시

이번에는 LLM에게 분석 질문이나 코드 초안을 요청하는 연습을 할 수 있습니다. API Key는 노트북에 직접 입력하지 않습니다.

분석 목적을 설명하고 필요한 코드를 요청하세요.

실습 과제

1. 이번 장에서 배운 내용을 바탕으로 분석 질문 3개를 작성합니다.
2. 그중 하나를 pandas 코드로 구현합니다.
3. 결과를 표 또는 그래프로 확인하고, LLM 도움을 받아 해석합니다.